# Practice Lab: Building AI-Powered UX (Lesson 12.12)

Module 12 . 4 exercises . Streaming + errors + progress + full chat app


# Lesson 12.12: Building AI-Powered UX

4 exercises: 2E + 1M + 1C

All exercises simulate UX behavior in pure Python.


In [ ]:
import random, json, time
random.seed(42)


---
## Exercise 1 (Easy): Streaming Demo


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json, random; random.seed(42)

class StreamSim:
    def __init__(self,ttft=150,tok_ms=50): self.ttft=ttft; self.tok=tok_ms; self.streams=[]
    def stream(self,prompt,response):
        words=response.split(); ttft=self.ttft+random.gauss(0,20); total=ttft
        events=[]
        for i,w in enumerate(words):
            events.append({"id":str(i+1),"event":"token","data":json.dumps({"token":w}),"ms":round(total,1)})
            total+=self.tok+random.gauss(0,5)
        events.append({"id":str(len(words)+1),"event":"done","data":json.dumps({"total":len(words)}),"ms":round(total,1)})
        r={"prompt":prompt,"tokens":len(words),"ttft":round(ttft,1),"total":round(total,1),"batch":round(total+500,1),"events":events}
        self.streams.append(r); return r

s=StreamSim(150,50)
print("Streaming Demo with TTFT:")
print(f"  {'Prompt':<18} {'Tok':>5} {'TTFT':>7} {'Stream':>8} {'Batch':>8} {'Speedup':>8}")
for p,r in [("Price?","GenAI course costs 14999 INR"),("Refund?","Full refund within 7 days 50% within 30"),
    ("Prereqs?","Python basics and high school math"),("Schedule?","Live Mon-Fri 7-8:30 PM IST recordings in 2 hours"),("Hello","Hi there")]:
    res=s.stream(p,r); sp=res["batch"]/res["ttft"]
    print(f"  {p:<18} {res['tokens']:>5} {res['ttft']:>6.0f}ms {res['total']:>7.0f}ms {res['batch']:>7.0f}ms {sp:>7.1f}x")
print(f"\n  SSE format (first 2 tokens):")
for e in s.streams[0]["events"][:2]: print(f"    id:{e['id']} event:{e['event']} data:{e['data']}")
avg_t=sum(x["ttft"] for x in s.streams)/len(s.streams)
print(f"\n  Avg TTFT: {avg_t:.0f}ms << 1000ms (Nielsen threshold)")


</details>


---
## Exercise 2 (Easy): Error States UX


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class ErrorUX:
    ERRORS={408:{"t":"Taking longer than usual","m":"The AI is thinking hard. Try again?","a":"Retry","ar":False},
        429:{"t":"Too many requests","m":"Great questions! Please wait a moment.","a":"Wait 10s","ar":True,"rs":10},
        500:{"t":"Something went wrong","m":"Our AI had a hiccup. Usually fixes itself.","a":"Retry","ar":False},
        403:{"t":"Can't answer that","m":"Outside what I can help with. Try rephrasing?","a":"Rephrase","ar":False},
        0:{"t":"You're offline","m":"Check your connection and try again.","a":"Retry","ar":False}}
    JARGON=["500","internal server error","exception","traceback","null","undefined","timeout","HTTP"]
    def get(self,code): return self.ERRORS.get(code,self.ERRORS[500])
    def check(self,msg):
        text=(msg["t"]+" "+msg["m"]).lower()
        return [j for j in self.JARGON if j.lower() in text]

ux=ErrorUX()
print("Error States UX:")
print(f"  {'Code':>5} {'Title':<28} {'Action':<10} {'Jargon'}")
clean=True
for code,scenario in [(408,"LLM slow"),(429,"Rate limited"),(500,"API down"),(403,"Content filter"),(0,"Offline")]:
    msg=ux.get(code); found=ux.check(msg)
    if found: clean=False
    print(f"  {code:>5} {msg['t']:<28} {msg['a']:<10} {'Clean' if not found else found}")
    print(f"        {msg['m']}")
    if msg.get("ar"): print(f"        Auto-retry in {msg['rs']}s")
print(f"\n  Jargon check: {'ALL CLEAN' if clean else 'FAILED'}")


</details>


---
## Exercise 3 (Medium): Agent Step-Progress


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random; random.seed(42)

class AgentProgress:
    def __init__(self,steps): self.steps=steps; self.snaps=[]
    def simulate(self):
        elapsed=0
        for i,step in enumerate(self.steps):
            states=[]
            for j in range(len(self.steps)):
                if j<i: states.append("[*]")
                elif j==i: states.append("[>]")
                else: states.append("[ ]")
            elapsed+=max(0.5,step["dur"]+random.uniform(-0.5,0.5))
            self.snaps.append({"t":round(elapsed,1),"dots":"".join(states),"msg":step["msg"]})
        self.snaps.append({"t":round(elapsed,1),"dots":"[*]"*len(self.steps),"msg":"Response ready!"})

steps=[{"id":"search","dur":2.0,"msg":"Searching knowledge base..."},
    {"id":"retrieve","dur":1.5,"msg":"Reading documents..."},
    {"id":"analyze","dur":3.0,"msg":"Analyzing information..."},
    {"id":"synthesize","dur":2.5,"msg":"Composing response..."},
    {"id":"cite","dur":1.0,"msg":"Adding citations..."},
    {"id":"format","dur":0.8,"msg":"Formatting answer..."}]

ap=AgentProgress(steps); ap.simulate()
print("Agent Step-Progress UI:")
print(f"  Legend: [*]=done [>]=active [ ]=pending")
print(f"  {'Time':>6}  {'Progress':<24} Status")
for s in ap.snaps:
    print(f"  {s['t']:>5.1f}s  {s['dots']:<24} {s['msg']}")
print(f"\n  Total: {ap.snaps[-1]['t']:.1f}s ({len(steps)} steps)")
print(f"  Without progress: user sees blank for {ap.snaps[-1]['t']:.0f}s")
print(f"  With progress: {len(steps)} engaging status updates")


</details>


---
## Exercise 4 (Challenge): Full Chat App Arch


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
print("Full Chat App Architecture:")
print("\n  12 Components:")
for c,d in [("ChatContainer","Root, conversation state"),("MessageList","History + auto-scroll"),
    ("MessageBubble","User/AI + confidence"),("StreamingText","Token-by-token SSE"),
    ("TypingIndicator","Animated dots"),("SkeletonLoader","Shimmer placeholder"),
    ("ErrorBanner","Friendly error + retry"),("FeedbackButtons","Thumbs up/down"),
    ("ConfidenceBadge","High/med/low score"),("SourceLinks","Clickable citations"),
    ("InputBar","Text + send + counter"),("AgentProgress","Step dots")]:
    print(f"    {c:<20} {d}")

print(f"\n  API Contract:")
print(f"    POST /v1/chat/stream -> SSE (token/meta/done events)")
print(f"    POST /v1/feedback -> {{response_id, type}}")
print(f"    GET  /v1/history?session_id=xxx")

costs=[("Cloud Run backend",30,60),("Vercel frontend",0,0),("Redis sessions",35,45),
    ("PostgreSQL feedback",10,15),("LLM APIs",50,150),("CDN",0,5)]
tl=sum(l for _,l,_ in costs); th=sum(h for _,_,h in costs)
print(f"\n  Monthly Cost (1000 daily users):")
for item,low,high in costs: print(f"    {item:<30} ${low}-${high}")
print(f"    {'TOTAL':<30} ${tl}-${th}")
print(f"    Per user: ${tl/1000:.3f}-${th/1000:.3f}/day")


</details>
